# VisionBridge — trained model check (Colab)

Inference/checking only. The base model is already trained. This notebook does not train or download the training dataset.

It verifies the checkpoint, runs a forward test, and then predicts from either an existing processed keypoint sample or uploaded `pose.npy` + `face.npy`.


## 1. Bootstrap the repository import path


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_ROOT = Path('/content/VisionBridge')
if not (REPO_ROOT / 'README.md').exists():
    subprocess.run(['git', 'clone', 'https://github.com/BharathWaj-K-R/VisionBridge.git', str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))
os.chdir(REPO_ROOT)
import app
print('Repository:', REPO_ROOT)
print('Backend import root:', BACKEND_ROOT)
print('APP IMPORT: PASS')


## 2. Locate the trained checkpoint and vocabulary

Expected:
`backend/app/models/weights/base_model.pt`
`backend/app/models/weights/base_model.vocab.json`


In [ ]:
import shutil
WEIGHTS = REPO_ROOT / 'backend/app/models/weights/base_model.pt'
VOCAB = REPO_ROOT / 'backend/app/models/weights/base_model.vocab.json'
if not WEIGHTS.exists() or not VOCAB.exists():
    from google.colab import files
    print('Upload BOTH base_model.pt and base_model.vocab.json')
    uploaded = files.upload()
    for name in ('base_model.pt', 'base_model.vocab.json'):
        if name not in uploaded:
            raise FileNotFoundError(f'Missing required artifact: {name}')
        shutil.copy(name, WEIGHTS.parent / name)
assert WEIGHTS.exists() and VOCAB.exists()
print(f'Weights: {WEIGHTS} ({WEIGHTS.stat().st_size/1e6:.2f} MB)')
print(f'Vocab:   {VOCAB} ({VOCAB.stat().st_size/1e3:.2f} KB)')


## 3. Validate the trained checkpoint


In [ ]:
import torch
from app.training.isltranslate import SimpleCharTokenizer
from app.models.base_model import load_frozen_base_model, POSE_INPUT_DIM, FACE_INPUT_DIM, MAX_SEQUENCE_LENGTH
tokenizer = SimpleCharTokenizer.load(VOCAB)
state = torch.load(WEIGHTS, map_location='cpu')
assert isinstance(state, dict) and 'output_head.weight' in state
checkpoint_vocab = int(state['output_head.weight'].shape[0])
assert checkpoint_vocab == tokenizer.vocab_size, f'Vocabulary mismatch: checkpoint={checkpoint_vocab}, tokenizer={tokenizer.vocab_size}'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size).to(device).eval()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable == 0, 'Base model is not frozen.'
print('Checkpoint vocabulary:', checkpoint_vocab)
print('Tokenizer vocabulary:', tokenizer.vocab_size)
print('Device:', device)
print('Trainable parameters:', trainable)
print('CHECKPOINT VALIDATION: PASS')


## 4. Forward-pass smoke test

Synthetic zeros verify only the input/output contract. They are not an accuracy test.


In [ ]:
frames = 16
pose = torch.zeros(1, frames, POSE_INPUT_DIM, device=device)
face = torch.zeros(1, frames, FACE_INPUT_DIM, device=device)
with torch.inference_mode():
    logits = model(pose, face)
print('Pose:', tuple(pose.shape))
print('Face:', tuple(face.shape))
print('Logits:', tuple(logits.shape))
assert logits.shape == (1, frames, tokenizer.vocab_size)
print('FORWARD TEST: PASS')


## 5. Find real keypoint input

A training dataset is **not required** for this model check. The notebook first looks for an existing processed sample. If none exists, the next cell will let you upload one real `pose.npy` + `face.npy` pair.


In [ ]:
from pathlib import Path
DATA_DIR = REPO_ROOT / 'data/processed/isltranslate'
processed_available = ((DATA_DIR/'ISLTranslate.csv').exists() and (DATA_DIR/'pose').exists() and (DATA_DIR/'face').exists() and any((DATA_DIR/'pose').glob('*.npy')) and any((DATA_DIR/'face').glob('*.npy')))
if processed_available:
    print('Processed keypoint data found. It will be used for the real inference check.')
else:
    print('No processed dataset found.')
    print('That is OK — you do NOT need the full dataset to test the trained model.')
    print('Use the next cell to upload one pose.npy + face.npy pair.')


## 6. Real model prediction — processed sample OR uploaded pose.npy + face.npy

This is the actual model check. It uses the same `decode_logits()` function used by VisionBridge inference.


In [ ]:
import numpy as np
from app.services.inference_service import decode_logits
from app.training.isltranslate import ISLTranslateKeypointDataset, _downsample_to_max_length

if processed_available:
    dataset = ISLTranslateKeypointDataset(DATA_DIR, tokenizer=tokenizer)
    assert len(dataset) > 0
    item = dataset[0]
    pose_t, face_t = _downsample_to_max_length(item['pose'], item['face'], item['uid'])
    truth = item['text']
    source = f'processed sample {item["uid"]}'
else:
    from google.colab import files
    print('Upload two files: pose.npy and face.npy')
    uploaded = files.upload()
    pose_candidates = [n for n in uploaded if n.lower().endswith('.npy') and 'pose' in n.lower()]
    face_candidates = [n for n in uploaded if n.lower().endswith('.npy') and 'face' in n.lower()]
    assert pose_candidates and face_candidates, 'Upload both pose.npy and face.npy.'
    pose_np = np.load(pose_candidates[0])
    face_np = np.load(face_candidates[0])
    assert pose_np.ndim == 2 and pose_np.shape[1] == POSE_INPUT_DIM, f'Expected pose shape [frames, {POSE_INPUT_DIM}], got {pose_np.shape}'
    assert face_np.ndim == 2 and face_np.shape[1] == FACE_INPUT_DIM, f'Expected face shape [frames, {FACE_INPUT_DIM}], got {face_np.shape}'
    assert pose_np.shape[0] == face_np.shape[0] and pose_np.shape[0] > 0, f'Pose/face frame mismatch: {pose_np.shape[0]} vs {face_np.shape[0]}'
    pose_t, face_t = _downsample_to_max_length(torch.from_numpy(pose_np).float(), torch.from_numpy(face_np).float(), 'uploaded')
    truth = None
    source = 'uploaded keypoints'

assert pose_t.ndim == 2 and pose_t.shape[1] == POSE_INPUT_DIM
assert face_t.ndim == 2 and face_t.shape[1] == FACE_INPUT_DIM
assert pose_t.shape[0] <= MAX_SEQUENCE_LENGTH

with torch.inference_mode():
    logits = model(pose_t.unsqueeze(0).to(device), face_t.unsqueeze(0).to(device))
prediction, confidence = decode_logits(logits)

print('SOURCE:', source)
print('POSE SHAPE:', tuple(pose_t.shape))
print('FACE SHAPE:', tuple(face_t.shape))
if truth is not None:
    print('GROUND TRUTH:', truth)
print('PREDICTED TEXT:', prediction)
print('CONFIDENCE:', round(float(confidence), 4))
print('MODEL LOGITS:', tuple(logits.shape))
print('REAL MODEL INFERENCE: PASS')


## Final result

PASS means the trained checkpoint loaded, the vocabulary matched, the model accepted the real keypoint sequence, and the VisionBridge decoder produced text.

A good prediction on one uploaded sample is a functional smoke test, not a benchmark. Accuracy requires a labeled held-out test set and CER/WER.
